Milestone 1

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 1. LOAD RAW DATA
df = pd.read_csv("azure_demand_raw.csv")

print("Initial shape:", df.shape)
print(df.head())

Initial shape: (5700, 10)
             timestamp        region service_type  usage_units  \
0  2023-03-12 01:14:50       us-west      Storage      8824.65   
1  2023-01-31 10:18:59  Europe-North      Compute     13098.88   
2  2023-08-30 21:38:21       us-east      Storage          NaN   
3  2024-02-19 05:23:30       us east      Storage      6408.14   
4  2024-03-19 21:40:00       us-west      Compute     11940.33   

   provisioned_capacity  cost_usd  availability_pct  cloud_spend_index  \
0                  8000   2823.89             99.94              92.91   
1                 10000   6287.46             99.69             103.90   
2                  8000       NaN             99.57              90.63   
3                  8000   2050.61             99.46             100.01   
4                 10000   5731.36             99.89              96.41   

   enterprise_demand_index  new_service_launch  
0                     1.21                   0  
1                     1.35        

In [ ]:
# 2. FIX TIMESTAMP
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])
df = df.sort_values('timestamp')

In [ ]:
# 3. REMOVE DUPLICATES
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (5500, 10)


In [ ]:
# 4. STANDARDIZE REGION NAMES
df['region'] = (
    df['region']
    .str.strip()
    .str.lower()
    .str.replace(" ", "-", regex=False)
)

df['region'] = df['region'].replace({
    'us-east': 'US-East',
    'us-west': 'US-West',
    'india-south': 'India-South',
    'europe-north': 'Europe-North'
})


In [ ]:
# 5. STANDARDIZE SERVICE TYPE
df['service_type'] = df['service_type'].str.title()

In [ ]:
# 6. HANDLE MISSING VALUES

# Time-series interpolation for usage
df['usage_units'] = df['usage_units'].interpolate(method='linear')

# Cost recompute if missing
df['cost_usd'] = df['cost_usd'].fillna(df['usage_units'] * 0.45)

# Availability forward fill
df['availability_pct'] = df['availability_pct'].ffill()

# Economic variables --> median
df['cloud_spend_index'] = df['cloud_spend_index'].fillna(
    df['cloud_spend_index'].median()
)

df['enterprise_demand_index'] = df['enterprise_demand_index'].fillna(
    df['enterprise_demand_index'].median()
)

# Technical binary variable
df['new_service_launch'] = df['new_service_launch'].fillna(0)


In [ ]:
# 7. VALIDATION RULES

# Remove unrealistic usage
df = df[df['usage_units'] > 0]

# Availability must be realistic
df = df[(df['availability_pct'] >= 95) & (df['availability_pct'] <= 100)]

# Remove negative costs
df = df[df['cost_usd'] > 0]

print("\nRemaining NaNs:\n", df.isnull().sum())
print("Final shape after cleaning:", df.shape)



Remaining NaNs:
 timestamp                  0
region                     0
service_type               0
usage_units                0
provisioned_capacity       0
cost_usd                   0
availability_pct           0
cloud_spend_index          0
enterprise_demand_index    0
new_service_launch         0
dtype: int64
Final shape after cleaning: (5500, 10)


In [ ]:
# 8. SAVE CLEAN DATA
df.to_csv("azure_demand_cleaned.csv", index=False)

print("Clean dataset saved successfully.")


Clean dataset saved successfully.


Milestone 2

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("azure_demand_cleaned.csv")

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(df.shape)
df.head()

(5500, 10)


,timestamp,region,service_type,usage_units,provisioned_capacity,cost_usd,availability_pct,cloud_spend_index,enterprise_demand_index,new_service_launch
0,2023-01-01 00:54:15,Europe-North,Compute,11070.03,10000,5313.61,99.60,110.15,1.14,0
1,2023-01-01 03:41:33,India-South,Compute,10606.77,10000,5091.25,99.84,98.46,1.24,0
2,2023-01-01 08:32:08,US-East,Storage,8647.21,8000,2767.11,99.57,108.16,1.07,1
3,2023-01-01 10:45:17,India-South,Compute,14125.23,10000,6780.11,99.91,128.31,1.24,0
4,2023-01-01 12:25:13,US-East,Compute,10985.42,10000,5273.00,99.80,102.13,1.27,0


In [3]:
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['week_of_year'] = df['timestamp'].dt.isocalendar().week.astype(int)
print(df)

               timestamp        region service_type  usage_units  \
0    2023-01-01 00:54:15  Europe-North      Compute     11070.03   
1    2023-01-01 03:41:33   India-South      Compute     10606.77   
2    2023-01-01 08:32:08       US-East      Storage      8647.21   
3    2023-01-01 10:45:17   India-South      Compute     14125.23   
4    2023-01-01 12:25:13       US-East      Compute     10985.42   
...                  ...           ...          ...          ...   
5495 2024-03-30 12:42:31       US-East      Compute      9205.24   
5496 2024-03-30 14:43:55   India-South      Storage     13582.78   
5497 2024-03-30 18:25:09   India-South      Compute     14645.86   
5498 2024-03-30 19:54:04       US-West      Storage     12837.44   
5499 2024-03-30 23:57:13       US-West      Compute     12670.51   

      provisioned_capacity   cost_usd  availability_pct  cloud_spend_index  \
0                    10000  5313.6100             99.60             110.15   
1                    10000 

In [4]:
df['lag_1_usage'] = df['usage_units'].shift(1)
df['lag_7_usage'] = df['usage_units'].shift(7)
df['lag_14_usage'] = df['usage_units'].shift(14)
print(df)

               timestamp        region service_type  usage_units  \
0    2023-01-01 00:54:15  Europe-North      Compute     11070.03   
1    2023-01-01 03:41:33   India-South      Compute     10606.77   
2    2023-01-01 08:32:08       US-East      Storage      8647.21   
3    2023-01-01 10:45:17   India-South      Compute     14125.23   
4    2023-01-01 12:25:13       US-East      Compute     10985.42   
...                  ...           ...          ...          ...   
5495 2024-03-30 12:42:31       US-East      Compute      9205.24   
5496 2024-03-30 14:43:55   India-South      Storage     13582.78   
5497 2024-03-30 18:25:09   India-South      Compute     14645.86   
5498 2024-03-30 19:54:04       US-West      Storage     12837.44   
5499 2024-03-30 23:57:13       US-West      Compute     12670.51   

      provisioned_capacity   cost_usd  availability_pct  cloud_spend_index  \
0                    10000  5313.6100             99.60             110.15   
1                    10000 

In [5]:
df['rolling_mean_3'] = df['usage_units'].rolling(window=3).mean()
df['rolling_mean_7'] = df['usage_units'].rolling(window=7).mean()

df['rolling_std_7'] = df['usage_units'].rolling(window=7).std()
print(df)

               timestamp        region service_type  usage_units  \
0    2023-01-01 00:54:15  Europe-North      Compute     11070.03   
1    2023-01-01 03:41:33   India-South      Compute     10606.77   
2    2023-01-01 08:32:08       US-East      Storage      8647.21   
3    2023-01-01 10:45:17   India-South      Compute     14125.23   
4    2023-01-01 12:25:13       US-East      Compute     10985.42   
...                  ...           ...          ...          ...   
5495 2024-03-30 12:42:31       US-East      Compute      9205.24   
5496 2024-03-30 14:43:55   India-South      Storage     13582.78   
5497 2024-03-30 18:25:09   India-South      Compute     14645.86   
5498 2024-03-30 19:54:04       US-West      Storage     12837.44   
5499 2024-03-30 23:57:13       US-West      Compute     12670.51   

      provisioned_capacity   cost_usd  availability_pct  cloud_spend_index  \
0                    10000  5313.6100             99.60             110.15   
1                    10000 

In [6]:
threshold = df['usage_units'].mean() + df['usage_units'].std()
df['demand_spike'] = (df['usage_units'] > threshold).astype(int)
print(df)

               timestamp        region service_type  usage_units  \
0    2023-01-01 00:54:15  Europe-North      Compute     11070.03   
1    2023-01-01 03:41:33   India-South      Compute     10606.77   
2    2023-01-01 08:32:08       US-East      Storage      8647.21   
3    2023-01-01 10:45:17   India-South      Compute     14125.23   
4    2023-01-01 12:25:13       US-East      Compute     10985.42   
...                  ...           ...          ...          ...   
5495 2024-03-30 12:42:31       US-East      Compute      9205.24   
5496 2024-03-30 14:43:55   India-South      Storage     13582.78   
5497 2024-03-30 18:25:09   India-South      Compute     14645.86   
5498 2024-03-30 19:54:04       US-West      Storage     12837.44   
5499 2024-03-30 23:57:13       US-West      Compute     12670.51   

      provisioned_capacity   cost_usd  availability_pct  cloud_spend_index  \
0                    10000  5313.6100             99.60             110.15   
1                    10000 

In [6]:
df = pd.get_dummies(df, columns=['region'], drop_first=True)

In [7]:
df['service_type'] = df['service_type'].map({
    'Compute': 1,
    'Storage': 0
})

In [8]:
df = df.dropna().reset_index(drop=True)
print("After lag/rolling cleanup:", df.shape)

After lag/rolling cleanup: (5486, 24)


In [9]:
features = [
    'usage_units',
    'provisioned_capacity',
    'cost_usd',
    'availability_pct',
    'cloud_spend_index',
    'enterprise_demand_index',
    'new_service_launch',
    'hour', 'day', 'day_of_week', 'month', 'week_of_year',
    'lag_1_usage', 'lag_7_usage', 'lag_14_usage',
    'rolling_mean_3', 'rolling_mean_7', 'rolling_std_7',
    'demand_spike'
]

target = 'usage_units'

In [10]:
df.to_csv("azure_demand_feature_engineered.csv", index=False)
print("Milestone-2 dataset saved successfully")

Milestone-2 dataset saved successfully
